# Milestone-3: RAG Pipeline

In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
kb = [str(row[row['answer']]) for _, row in train.iterrows()]

model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

In [ ]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_150 = train.iloc[150]
labels_150 = [str(row_150[l]) for l in ['A', 'B', 'C', 'D', 'E']]
res = zs(str(row_150['prompt']), candidate_labels=labels_150)
idx = res['labels'].index(str(row_150[row_150['answer']]))
print("Ground truth prob:", res['scores'][idx]) # 0.384

In [ ]:
p_emb = model.encode([str(row_150['prompt'])])
D, I = index.search(p_emb, 10) 
print("Rank of index 150:", list(I[0]).index(150) + 1) # 10